In [3]:
import tensorflow as tf
print(tf.__version__)

from google.colab import drive
import os

# mounting my drive
drive.mount('/content/drive')

# making a folder in my drive for my projectt
project_directory = "/content/drive/MyDrive/ECommerce_Project"
os.makedirs(project_directory, exist_ok=True)


import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

#import requests
#import shutil

from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix

import tensorflow as tf
from tensorflow import keras
from tensorflow.keras.models import Sequential
from tensorflow.keras import layers, models

from tensorflow.keras.applications import MobileNetV2
from tensorflow.keras.applications.mobilenet_v2 import preprocess_input

2.20.0
Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [4]:
df = pd.read_csv('/content/electronics_product.csv')
# removing rupee symbols and the commas
df['price_no_symbols'] = df['actual_price'].astype(str).str.replace('₹', '', regex=False)
df['price_no_symbols'] = df['price_no_symbols'].str.replace(',', '', regex=False)

# changing it to the numbers
df['clean_price_inr'] = pd.to_numeric(df['price_no_symbols'], errors='coerce')
df = df.dropna(subset=['clean_price_inr', 'image'])

# changing the price to us dollars
df['price_usd'] = df['clean_price_inr'] / 96.08
df['price_usd'] = df['price_usd'].round(2)


low_cutoff_usd = df['price_usd'].quantile(0.20)
high_cutoff_usd = df['price_usd'].quantile(0.80)

# separating budget and premium
budget_df = df[df['price_usd'] <= low_cutoff_usd].copy()
budget_df['label'] = 0  # Standard / Budget

premium_df = df[df['price_usd'] >= high_cutoff_usd].copy()
premium_df['label'] = 1  # Premium


# filtering out the middle rows
filtered_df = df[(df['price_usd'] <= low_cutoff_usd) | (df['price_usd'] >= high_cutoff_usd)].reset_index(drop=True)

filtered_df = filtered_df.drop_duplicates(subset=['image']).reset_index(drop=True)

def assign_label(price):
    if price >= high_cutoff_usd:
        return 1
    else:
        return 0

filtered_df['label'] = filtered_df['price_usd'].apply(assign_label)

filtered_df.head()

filtered_df['label'].dtype


dtype('int64')

In [5]:
# downloading images - i'm commenting out since images are already saved

# if os.path.exists("Premium"):
#     shutil.rmtree("Premium")
# if os.path.exists("Standard"):
#     shutil.rmtree("Standard")

# os.makedirs("Premium", exist_ok=True)
# os.makedirs("Standard", exist_ok=True)

# download loop
# for index, row in filtered_df.iterrows():
     #url = row["image"]
     #label = row["label"]

#     try:
#         response = requests.get(url, timeout=10)
#         if response.status_code == 200:
#             folder = "Premium" if label == 1 else "Standard"
#             filename = f"{folder}/{index}.jpg"
#
#             with open(filename, "wb") as f:
#                 f.write(response.content)
#     except:       # using to skip the broken links
#         pass

In [6]:
from google.colab import drive
import shutil
import os
# commenting out because i already made the folder

#drive.mount('/content/drive')

#drive_dir = '/content/drive/MyDrive/ECommerce_Project'
#os.makedirs(f'{drive_dir}/Premium', exist_ok=True)
#os.makedirs(f'{drive_dir}/Standard', exist_ok=True)

#backing up the downloaded images to myGoogle Drive
#shutil.copytree('/content/Premium', f'{drive_dir}/Premium', dirs_exist_ok=True)
#shutil.copytree('/content/Standard', f'{drive_dir}/Standard', dirs_exist_ok=True)


In [7]:
# building the model yay!
data ='/content/drive/MyDrive/ECommerce_Project'


image_size = (224, 224)
batch_size = 32

train_dataset = tf.keras.utils.image_dataset_from_directory(
    data,
    validation_split=0.2,
    subset='training',
    seed=42, # for the randomly shuffling
    image_size=image_size,
    batch_size=batch_size
)

validation_dataset = tf.keras.utils.image_dataset_from_directory(
    data,
    validation_split=0.2,
    subset='validation',
    seed=42,
    image_size=image_size,
    batch_size=batch_size
)

base_model = MobileNetV2(input_shape=(224, 224, 3), include_top=False, weights='imagenet')
#I am freezing the mode layers so i don't overwrite the pretrained weights
base_model.trainable = False

model = Sequential([
    layers.Lambda(preprocess_input, input_shape=(224, 224, 3)),
    base_model,
    layers.GlobalAveragePooling2D(),
    layers.Dropout(0.2), # this will stop the overfitting
    layers.Dense(1, activation='sigmoid') #like the fully connected, sigmoid activation makes it between 0 and 1
])

#compiling the model
model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=0.0001), #adam adjusts the model weights.
    loss='binary_crossentropy', # loss function
    metrics=['accuracy']
)

model.summary()


Found 3689 files belonging to 2 classes.
Using 2952 files for training.
Found 3689 files belonging to 2 classes.
Using 737 files for validation.
9406464/9406464 ━━━━━━━━━━━━━━━━━━━━ 2s 0us/step


/usr/local/lib/python3.12/dist-packages/keras/src/layers/core/lambda_layer.py:65: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ lambda (Lambda)                 │ (None, 224, 224, 3)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ mobilenetv2_1.00_224            │ (None, 7, 7, 1280)     │     2,257,984 │
│ (Functional)                    │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_average_pooling2d        │ (None, 1280)           │             0 │
│ (GlobalAveragePooling2D)        │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 1280)           │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 1)              │         1,281 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 2,259,265 (8.62 MB)

 Trainable params: 1,281 (5.00 KB)

 Non-trainable params: 2,257,984 (8.61 MB)

In [8]:
epochs = 10

results = model.fit(train_dataset, validation_data=validation_dataset, epochs=epochs)

print('it finished training.')

Epoch 1/10
93/93 ━━━━━━━━━━━━━━━━━━━━ 1458s 15s/step - accuracy: 0.5549 - loss: 0.7397 - val_accuracy: 0.6079 - val_loss: 0.6675
Epoch 2/10
93/93 ━━━━━━━━━━━━━━━━━━━━ 21s 223ms/step - accuracy: 0.6128 - loss: 0.6639 - val_accuracy: 0.6540 - val_loss: 0.6200
Epoch 3/10
93/93 ━━━━━━━━━━━━━━━━━━━━ 18s 198ms/step - accuracy: 0.6667 - loss: 0.6131 - val_accuracy: 0.6825 - val_loss: 0.5878
Epoch 4/10
93/93 ━━━━━━━━━━━━━━━━━━━━ 20s 214ms/step - accuracy: 0.6985 - loss: 0.5764 - val_accuracy: 0.7056 - val_loss: 0.5654
Epoch 5/10
93/93 ━━━━━━━━━━━━━━━━━━━━ 20s 220ms/step - accuracy: 0.7114 - loss: 0.5480 - val_accuracy: 0.7137 - val_loss: 0.5483
Epoch 6/10
93/93 ━━━━━━━━━━━━━━━━━━━━ 20s 218ms/step - accuracy: 0.7412 - loss: 0.5258 - val_accuracy: 0.7327 - val_loss: 0.5353
Epoch 7/10
93/93 ━━━━━━━━━━━━━━━━━━━━ 19s 203ms/step - accuracy: 0.7483 - loss: 0.5104 - val_accuracy: 0.7395 - val_loss: 0.5245
Epoch 8/10
93/93 ━━━━━━━━━━━━━━━━━━━━ 20s 216ms/step - accuracy: 0.7575 - loss: 0.4976 - val_accu